Read Silver Table 

In [0]:
# Read Silver Table
silver_df = spark.table("workspace.silver.superstore_orders_silver")

display(silver_df)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_customer = (
    silver_df
    .select(
        "customer_id",
        "customer_name",
        "segment"
    )
    .dropDuplicates()
)

In [0]:
dim_customer = dim_customer.withColumn(
    "customer_key",
    monotonically_increasing_id()
)

In [0]:
from pyspark.sql.functions import col

dim_customer = dim_customer.select(
    "customer_key",
    "customer_id",
    "customer_name",
    "segment"
)

In [0]:
display(dim_customer)

In [0]:
dim_customer.write \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_customer")

Creation of dim_products

In [0]:
from pyspark.sql.functions import first
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dim_product = (
    silver_df
    .groupBy("product_id")
    .agg(
        first("product_name").alias("product_name"),
        first("category").alias("category"),
        first("sub_category").alias("sub_category")
    )
)

window_spec = Window.orderBy("product_id")

dim_product = (
    dim_product
    .withColumn("product_key", row_number().over(window_spec))
    .select(
        "product_key",
        "product_id",
        "product_name",
        "category",
        "sub_category"
    )
)

In [0]:
dim_product.write \
.mode("overwrite") \
.saveAsTable("workspace.gold.dim_product")

Create dim_location

In [0]:
window_spec = Window.orderBy(
    "country_region",
    "state_province",
    "city",
    "postal_code"
)

dim_location = (
    silver_df
    .select(
        "country_region",
        "state_province",
        "city",
        "postal_code",
        "region"
    )
    .dropDuplicates()
    .withColumn(
        "location_key",
        row_number().over(window_spec)
    )
    .select(
        "location_key",
        "country_region",
        "state_province",
        "city",
        "postal_code",
        "region"
    )
)

display(dim_location)

In [0]:
dim_location.write \
.mode("overwrite") \
.saveAsTable("workspace.gold.dim_location")

Create dim_date

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    quarter,
    dayofmonth,
    dayofweek,
    date_format
)

dim_date = (
    silver_df
    .select("order_date")
    .dropDuplicates()
)

window_spec = Window.orderBy("order_date")

dim_date = (
    dim_date
    .withColumn("date_key", row_number().over(window_spec))
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("quarter", quarter("order_date"))
    .withColumn("day", dayofmonth("order_date"))
    .withColumn("weekday", date_format("order_date", "EEEE"))
)

display(dim_date)

In [0]:
dim_date.write \
.mode("overwrite") \
.saveAsTable("workspace.gold.dim_date")

Read Dimention all Tables 

In [0]:
dim_customer = spark.table("workspace.gold.dim_customer")

dim_product = spark.table("workspace.gold.dim_product")

dim_location = spark.table("workspace.gold.dim_location")

dim_date = spark.table("workspace.gold.dim_date")

Building Fact table

In [0]:
fact_sales = (
    silver_df

    .join(
        dim_customer.select("customer_key", "customer_id"),
        on="customer_id",
        how="left"
    )

    .join(
        dim_product.select("product_key", "product_id"),
        on="product_id",
        how="left"
    )

    .join(
        dim_location.select(
            "location_key",
            "country_region",
            "state_province",
            "city",
            "postal_code",
            "region"
        ),
        on=[
            "country_region",
            "state_province",
            "city",
            "postal_code",
            "region"
        ],
        how="left"
    )

    .join(
        dim_date.select("date_key", "order_date"),
        on="order_date",
        how="left"
    )
)

In [0]:
fact_sales = fact_sales.select(

    "order_id",

    "customer_key",

    "product_key",

    "location_key",

    "date_key",

    "sales",

    "quantity",

    "discount",

    "profit"

)

In [0]:
display(fact_sales)

In [0]:
fact_sales.write \
.mode("overwrite") \
.saveAsTable("workspace.gold.fact_sales")